# Phase 6 Explainability

Do both models rely on drivers a credit officer would recognise and defend?

After what this project found about endogeneity, this is a substantive check. Two models,
two different failure modes:

- **GBM (champion, application-only)** is opaque. SHAP gives per-prediction attributions;
  aggregating them gives a global ranking, and correlating each feature against its own SHAP
  value gives the direction the model learned. A driver pointing the wrong way is a red flag
  no accuracy metric would ever surface.
- **Scorecard** is transparent but its coefficients are in WOE units, which nobody outside
  this repo reads. Section 4 converts them into a points table - what each answer on the
  application form is actually worth.

SHAP is run on the champion, not the full-pool model: there it would report that `sub_grade`
dominates, which is already known and explains nothing.


In [ ]:
from pathlib import Path

import polars as pl
import yaml

from credit_risk.data.ingestion import load_raw_accepted_loans
from credit_risk.data.target import build_target
from credit_risk.explainability.attribution import (
    compute_shap_values,
    points_range,
    rank_agreement,
    scorecard_points,
    shap_direction_report,
    shap_global_importance,
)
from credit_risk.features.build_dataset import (
    APPLICATION_FEATURES,
    application_features,
    assemble_feature_matrix,
)
from credit_risk.models.gbm import prepare_lgb_frame, train_gbm
from credit_risk.models.scorecard import train_scorecard

pl.Config.set_tbl_rows(80)
CONFIG_PATH = Path("../configs/base.yaml")
DATA_PATH = Path("../data/raw/accepted_2007_to_2018Q4.csv")

final = assemble_feature_matrix(
    build_target(load_raw_accepted_loans(DATA_PATH), CONFIG_PATH), CONFIG_PATH
)
splits = {n: final.filter(pl.col("split") == n) for n in ("train", "validation", "oot_test")}
print({n: df.height for n, df in splits.items()})


In [ ]:
features = application_features(final)
params = yaml.safe_load(Path("../configs/gbm_best_params_application.yaml").read_text())
gbm, features = train_gbm(splits["train"], splits["validation"], params=params, features=features)
print(f"champion trained on {len(features)} features, {gbm.best_iteration} iterations")


## 1. What the GBM actually uses

SHAP values are computed on OOT, not train: the question is what drives the model on data it
has never seen. Values are in log-odds space, so they sum to the raw margin.

`share` is each feature's portion of total attribution. Watch for concentration - if two or
three features carry most of it, the other sixty are decoration and the model is simpler than
its feature count suggests.


In [ ]:
oot_frame = prepare_lgb_frame(splits["oot_test"], features)
shap_values, sampled = compute_shap_values(gbm, oot_frame, sample_size=20_000)

importance = shap_global_importance(shap_values, sampled)
print(importance.head(25).to_pandas().to_string(index=False))
print(f"\ntop 5 carry {importance['share'].head(5).sum():.1%} of total attribution")
print(f"top 15 carry {importance['share'].head(15).sum():.1%}")
importance.write_csv("../docs/shap_importance.csv")


## 2. Direction the check that matters

`direction` is Spearman between a feature's value and its own SHAP contribution.
`raises_risk = True` means higher values push predicted risk up.

Read the top features against what a credit officer would expect: higher `annual_inc` should
LOWER risk, higher `dti` should RAISE it, higher `fico_range_low` should LOWER it, more
recent inquiries should RAISE it. Any reversal among the high-importance features needs an
explanation before this model goes anywhere near serving.

Categoricals are reported as null rather than guessed: their codes have no order.


In [ ]:
direction = shap_direction_report(shap_values, sampled)
review = (
    importance.join(direction, on="feature")
    .head(20)
    .select("feature", "mean_abs_shap", "share", "direction", "raises_risk")
)
print(review.to_pandas().to_string(index=False))
direction.write_csv("../docs/shap_direction.csv")


In [ ]:
# Explicit expectations, so the check is a test rather than an eyeball.
EXPECTED_RAISES_RISK = {
    "dti": True, "annual_inc": False, "fico_range_low": False,
    "mths_since_recent_inq": False,   # more months SINCE an inquiry is safer
    "acc_open_past_24mths": True, "percent_bc_gt_75": True,
    "term_months": True, "tot_hi_cred_lim": False, "bc_open_to_buy": False,
}
observed = dict(zip(*direction[["feature", "raises_risk"]], strict=True))
for feature, expected in EXPECTED_RAISES_RISK.items():
    got = observed.get(feature)
    verdict = "ok" if got == expected else ("UNKNOWN" if got is None else "CONTRADICTS EXPECTATION")
    print(f"{feature:<24} expected_raises_risk={str(expected):<5} observed={str(got):<5} {verdict}")


## 3. SHAP versus IV where the GBM earns its lift

IV is univariate; SHAP is not. Low agreement means the GBM is finding structure a
single-variable screen cannot see.

This is the expected explanation for a specific measured result: the GBM beats the linear
scorecard by +0.0122 AUC with `sub_grade` present but +0.0294 without it. With `sub_grade`
gone, the interactions have to be rediscovered, and apparently they are.


In [ ]:
iv = pl.read_csv("../docs/iv_ranking_full.csv").filter(pl.col("iv").is_not_null())
print(rank_agreement(importance, iv))

# Features the GBM leans on that the IV screen rated as unusable (< 0.02).
weak_iv = set(iv.filter(pl.col("iv") < 0.02)["feature"].to_list())
missed = importance.head(20).filter(pl.col("feature").is_in(weak_iv))
print("\nhigh SHAP but screened out by IV:")
print(missed.to_pandas().to_string(index=False) if missed.height else "  none")


## 4. The scorecard as a points table

This is the scorecard deliverable. Coefficients on WOE units are not usable by anyone; a
points lookup is.

    factor = pdo / ln(2),  offset = base_score - factor * ln(base_odds)
    points(feature, bin) = -factor * coefficient * WOE + (offset - factor * intercept) / n

Correctly signed coefficients are negative and higher WOE means safer, so safer bins earn
more points. Rows sum to the applicant's total score.


In [ ]:
sc_model, sc_encoder = train_scorecard(splits["train"], features=APPLICATION_FEATURES)
points = scorecard_points(sc_encoder, sc_model, APPLICATION_FEATURES, pdo=20, base_score=600, base_odds=50.0)

print(points.filter(pl.col("feature").is_in(["fico_range_low", "dti", "term_months"]))
      .select("feature", "bin", "n", "bad_rate", "woe", "points")
      .to_pandas().to_string(index=False))
points.write_csv("../docs/scorecard_points.csv")


## 5. Which features actually move a score

`swing` is the points difference between a feature's worst and best bin. A feature with a
large coefficient but bins that barely differ moves nobody's score, so this - not the
coefficient list - is the ranking to show a credit committee.

Compare it against the SHAP ranking from section 1. Broad agreement means both models read
the population the same way, which is the reassuring outcome. Sharp disagreement is worth
understanding before either is trusted.


In [ ]:
swing = points_range(points)
print(swing.to_pandas().to_string(index=False))

print("\nagreement with SHAP ranking:", rank_agreement(
    importance, swing.rename({"swing": "value"})
))
